In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Importing Data

In [28]:
# Define file paths
races_path = 'data/f1_dataset/races.csv'
results_path = 'data/f1_dataset/results.csv'
drivers_path = 'data/f1_dataset/drivers.csv'
constructors_path = 'data/f1_dataset/constructors.csv'
sprints_path = 'data/f1_dataset/sprint_results.csv'
qualifying_path = 'data/f1_dataset/qualifying.csv'
driver_standings_path = 'data/f1_dataset/driver_standings.csv'
constructor_standings_path = 'data/f1_dataset/constructor_standings.csv'
lap_times_path = 'data/f1_dataset/lap_times.csv'
pit_stops_path = 'data/f1_dataset/pit_stops.csv'
circuits_path = 'data/f1_dataset/circuits.csv'

# Load races dataframe
races_df = pd.read_csv(races_path)
# Filter races for years 2022 to present
races_2022_present = races_df[races_df['year'] >= 2022]
races_2022_present.reset_index(drop=True, inplace=True)
# Get the list of race IDs for the filtered period
race_ids = races_2022_present['raceId'].unique()

# Load results dataframe
results_df = pd.read_csv(results_path)
# Filter results based on the race IDs
results_2022_present = results_df[results_df['raceId'].isin(race_ids)]
results_2022_present.reset_index(drop=True, inplace=True)
#Load drivers and constructors for 2022-Present
drivers_2022_present = results_2022_present['driverId'].unique()
constructors_2022_present = results_2022_present['constructorId'].unique()

#Load sprints and qualifying for 2022-Present
sprints_df = pd.read_csv(sprints_path)
qualifying_df = pd.read_csv(qualifying_path)
sprints_2022_present = sprints_df[sprints_df['raceId'].isin(race_ids)]
sprints_2022_present.reset_index(drop=True, inplace=True)
qualifying_2022_present = qualifying_df[qualifying_df['raceId'].isin(race_ids)]
qualifying_2022_present.reset_index(drop=True, inplace=True)

lap_times_df = pd.read_csv(lap_times_path)
lap_times_2022_present = lap_times_df[lap_times_df['raceId'].isin(race_ids)]
lap_times_2022_present.reset_index(drop=True, inplace=True)

pit_stops_df = pd.read_csv(pit_stops_path)
pit_stops_2022_present = pit_stops_df[pit_stops_df['raceId'].isin(race_ids)]
pit_stops_2022_present.reset_index(drop=True, inplace=True)

# Load additional context dataframes (no filtering needed usually)
constructors_df = pd.read_csv(constructors_path)
constructors_2022_present_df = constructors_df[constructors_df['constructorId'].isin(constructors_2022_present)]
constructors_2022_present_df.reset_index(drop=True, inplace=True)

circuits_df = pd.read_csv(circuits_path)
circuits_2022_present_df = circuits_df[circuits_df['circuitId'].isin(races_2022_present['circuitId'].unique())]
circuits_2022_present_df.reset_index(drop=True, inplace=True)

drivers_df = pd.read_csv(drivers_path)
drivers_2022_present_df = drivers_df[drivers_df['driverId'].isin(drivers_2022_present)]
drivers_2022_present_df.reset_index(drop=True, inplace=True)


# Display the first few rows and shapes to verify
print("Races (2022-Present) Shape:", races_2022_present.shape)
print("Results (2022-Present) Shape:", results_2022_present.shape)
print("Sprints Results (2022-Present) Shape:", sprints_2022_present.shape)
print("Qualifying (2022-Present) Shape:", qualifying_2022_present.shape)
print("Driver (2022-Present) Shape:", drivers_2022_present.shape)
print("Constructor (2022-Present) Shape:", constructors_2022_present.shape)
print("Lap Times (2022-Present) Shape:", lap_times_2022_present.shape)
print("Pit Stops (2022-Present) Shape:", pit_stops_2022_present.shape)
print("Circuits (2022-Present) Shape:", circuits_2022_present_df.shape)
print("Constructor (2022-Present) Shape:", constructors_2022_present_df.shape)


Races (2022-Present) Shape: (68, 18)
Results (2022-Present) Shape: (1359, 18)
Sprints Results (2022-Present) Shape: (300, 16)
Qualifying (2022-Present) Shape: (1359, 9)
Driver (2022-Present) Shape: (28,)
Constructor (2022-Present) Shape: (12,)
Lap Times (2022-Present) Shape: (74489, 6)
Pit Stops (2022-Present) Shape: (2542, 7)
Circuits (2022-Present) Shape: (25, 9)
Constructor (2022-Present) Shape: (12, 5)


## Feature Selection

In [15]:
results_2022_present.info()
sprints_2022_present.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1359 entries, 0 to 1358
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         1359 non-null   int64  
 1   raceId           1359 non-null   int64  
 2   driverId         1359 non-null   int64  
 3   constructorId    1359 non-null   int64  
 4   number           1359 non-null   object 
 5   grid             1359 non-null   int64  
 6   position         1359 non-null   object 
 7   positionText     1359 non-null   object 
 8   positionOrder    1359 non-null   int64  
 9   points           1359 non-null   float64
 10  laps             1359 non-null   int64  
 11  time             1359 non-null   object 
 12  milliseconds     1359 non-null   object 
 13  fastestLap       1359 non-null   object 
 14  rank             1359 non-null   object 
 15  fastestLapTime   1359 non-null   object 
 16  fastestLapSpeed  1359 non-null   object 
 17  statusId      

In [ ]:
qualifying_2022_present.info()

In [5]:
circuits_2022_present_df.info()
circuits_2022_present_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   circuitId   25 non-null     int64  
 1   circuitRef  25 non-null     object 
 2   name        25 non-null     object 
 3   location    25 non-null     object 
 4   country     25 non-null     object 
 5   lat         25 non-null     float64
 6   lng         25 non-null     float64
 7   alt         25 non-null     int64  
 8   url         25 non-null     object 
dtypes: float64(2), int64(2), object(5)
memory usage: 1.9+ KB


,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
2,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.5700,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
3,6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
4,7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5000,-73.52280,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Vi...


In [ ]:
constructors_2022_present_df.info()
constructors_2022_present_df.head()

In [ ]:
races_2022_present.info()
races_2022_present.head()

In [6]:
drivers_2022_present_df.info()
drivers_2022_present_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     28 non-null     int64 
 1   driverRef    28 non-null     object
 2   number       28 non-null     object
 3   code         28 non-null     object
 4   forename     28 non-null     object
 5   surname      28 non-null     object
 6   dob          28 non-null     object
 7   nationality  28 non-null     object
 8   url          28 non-null     object
dtypes: int64(1), object(8)
memory usage: 2.1+ KB


,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
2,20,vettel,5,VET,Sebastian,Vettel,1987-07-03,German,http://en.wikipedia.org/wiki/Sebastian_Vettel
3,842,gasly,10,GAS,Pierre,Gasly,1996-02-07,French,http://en.wikipedia.org/wiki/Pierre_Gasly
4,807,hulkenberg,27,HUL,Nico,Hülkenberg,1987-08-19,German,http://en.wikipedia.org/wiki/Nico_H%C3%BClkenberg


In [50]:
results_features = ['raceId','constructorId', 'grid', 'positionOrder', 'points', 'milliseconds', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'session_type']
circuits_features = ['circuitId', 'circuitRef', 'alt', 'country']
constructors_features = ['constructorId', 'constructorRef']
races_features = ['raceId', 'circuitId']
#qualifying_features = ['raceId','constructorId', 'position', 'q1', 'q2', 'q3']

In [77]:
results_df = results_2022_present.copy()
sprint_df = sprints_2022_present.copy()
results_df['session_type'] = 'race'
sprint_df['session_type'] = 'sprint'

combined_df = pd.concat([results_df, sprint_df], ignore_index=True)
combined_df

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,session_type
0,25406,1074,844,6,16,1,1,1,1,26.0,57,1:37:33.584,5853584,51,1,1:34.570,206.018,1,race
1,25407,1074,832,6,55,3,2,2,2,18.0,57,+5.598,5859182,52,3,1:35.740,203.501,1,race
2,25408,1074,1,131,44,5,3,3,3,15.0,57,+9.675,5863259,53,5,1:36.228,202.469,1,race
3,25409,1074,847,131,63,9,4,4,4,12.0,57,+11.211,5864795,56,6,1:36.302,202.313,1,race
4,25410,1074,825,210,20,7,5,5,5,10.0,57,+14.754,5868338,53,8,1:36.623,201.641,1,race
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1654,356,1143,859,215,30,10,16,16,16,0.0,19,+34.356,1657366,18,NaN,1:25.762,NaN,1,sprint
1655,357,1143,852,215,22,16,17,17,17,0.0,19,+35.102,1658112,17,NaN,1:25.838,NaN,1,sprint
1656,358,1143,861,3,43,0,18,18,18,0.0,19,+35.639,1658649,18,NaN,1:25.599,NaN,1,sprint
1657,359,1143,855,15,24,18,19,19,19,0.0,19,+1:11.436,1694446,19,NaN,1:25.051,NaN,1,sprint


In [78]:
con1 = pd.merge(combined_df[results_features], races_2022_present[races_features], on='raceId')
con2 = pd.merge(con1, constructors_df[constructors_features], on='constructorId')
con3 = pd.merge(con2, circuits_2022_present_df[circuits_features], on='circuitId')
selected_df = con3.drop(['constructorId', 'circuitId'], axis=1)


In [79]:
selected_df.info()
selected_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1659 entries, 0 to 1658
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   raceId           1659 non-null   int64  
 1   grid             1659 non-null   int64  
 2   positionOrder    1659 non-null   int64  
 3   points           1659 non-null   float64
 4   milliseconds     1659 non-null   object 
 5   fastestLapTime   1659 non-null   object 
 6   fastestLapSpeed  1359 non-null   object 
 7   statusId         1659 non-null   int64  
 8   session_type     1659 non-null   object 
 9   constructorRef   1659 non-null   object 
 10  circuitRef       1659 non-null   object 
 11  alt              1659 non-null   int64  
 12  country          1659 non-null   object 
dtypes: float64(1), int64(5), object(7)
memory usage: 168.6+ KB


,raceId,grid,positionOrder,points,milliseconds,fastestLapTime,fastestLapSpeed,statusId,session_type,constructorRef,circuitRef,alt,country
0,1074,1,1,26.0,5853584,1:34.570,206.018,1,race,ferrari,bahrain,7,Bahrain
1,1074,3,2,18.0,5859182,1:35.740,203.501,1,race,ferrari,bahrain,7,Bahrain
2,1074,5,3,15.0,5863259,1:36.228,202.469,1,race,mercedes,bahrain,7,Bahrain
3,1074,9,4,12.0,5864795,1:36.302,202.313,1,race,mercedes,bahrain,7,Bahrain
4,1074,7,5,10.0,5868338,1:36.623,201.641,1,race,haas,bahrain,7,Bahrain


In [80]:
selected_df.isna().sum()

raceId               0
grid                 0
positionOrder        0
points               0
milliseconds         0
fastestLapTime       0
fastestLapSpeed    300
statusId             0
session_type         0
constructorRef       0
circuitRef           0
alt                  0
country              0
dtype: int64

In [81]:
selected_df = selected_df.dropna()

In [82]:
selected_df.describe().T

,count,mean,std,min,25%,50%,75%,max
raceId,1359.0,1109.534216,20.755186,1074.0,1091.5,1110.0,1127.5,1144.0
grid,1359.0,9.947020,5.819080,0.0,5.0,10.0,15.0,20.0
positionOrder,1359.0,10.493010,5.764760,1.0,5.5,10.0,15.0,20.0
points,1359.0,5.097130,7.245494,0.0,0.0,1.0,9.5,26.0
statusId,1359.0,8.894776,23.371432,1.0,1.0,1.0,11.0,141.0
alt,1359.0,250.029433,482.924906,-7.0,7.0,37.0,264.0,2227.0


## Converting Data Types

In [83]:
colnames = {'grid': "start_position", 'positionOrder': 'finish_position', 'milliseconds': 'time_in_mil', 'fastestLapSpeed': 'max_speed_mph'}
selected_df.rename(columns=colnames, inplace=True)
selected_df

,raceId,start_position,finish_position,points,time_in_mil,fastestLapTime,max_speed_mph,statusId,session_type,constructorRef,circuitRef,alt,country
0,1074,1,1,26.0,5853584,1:34.570,206.018,1,race,ferrari,bahrain,7,Bahrain
1,1074,3,2,18.0,5859182,1:35.740,203.501,1,race,ferrari,bahrain,7,Bahrain
2,1074,5,3,15.0,5863259,1:36.228,202.469,1,race,mercedes,bahrain,7,Bahrain
3,1074,9,4,12.0,5864795,1:36.302,202.313,1,race,mercedes,bahrain,7,Bahrain
4,1074,7,5,10.0,5868338,1:36.623,201.641,1,race,haas,bahrain,7,Bahrain
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1354,1144,14,16,0.0,\N,1:25.637,222.002,11,race,haas,yas_marina,3,UAE
1355,1144,12,17,0.0,\N,1:28.751,214.212,5,race,rb,yas_marina,3,UAE
1356,1144,9,18,0.0,\N,1:29.482,212.462,130,race,sauber,yas_marina,3,UAE
1357,1144,20,19,0.0,\N,1:29.411,212.631,5,race,williams,yas_marina,3,UAE


In [84]:
selected_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1359 entries, 0 to 1358
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   raceId           1359 non-null   int64  
 1   start_position   1359 non-null   int64  
 2   finish_position  1359 non-null   int64  
 3   points           1359 non-null   float64
 4   time_in_mil      1359 non-null   object 
 5   fastestLapTime   1359 non-null   object 
 6   max_speed_mph    1359 non-null   object 
 7   statusId         1359 non-null   int64  
 8   session_type     1359 non-null   object 
 9   constructorRef   1359 non-null   object 
 10  circuitRef       1359 non-null   object 
 11  alt              1359 non-null   int64  
 12  country          1359 non-null   object 
dtypes: float64(1), int64(5), object(7)
memory usage: 148.6+ KB


In [85]:
cols = ['time_in_mil', 'max_speed_mph']
for i in cols:
    selected_df[i] = pd.to_numeric(selected_df[i], errors='coerce')

selected_df = selected_df.fillna(0) #means the driver was not able to complete the race or did not to the race as all

selected_df


,raceId,start_position,finish_position,points,time_in_mil,fastestLapTime,max_speed_mph,statusId,session_type,constructorRef,circuitRef,alt,country
0,1074,1,1,26.0,5853584.0,1:34.570,206.018,1,race,ferrari,bahrain,7,Bahrain
1,1074,3,2,18.0,5859182.0,1:35.740,203.501,1,race,ferrari,bahrain,7,Bahrain
2,1074,5,3,15.0,5863259.0,1:36.228,202.469,1,race,mercedes,bahrain,7,Bahrain
3,1074,9,4,12.0,5864795.0,1:36.302,202.313,1,race,mercedes,bahrain,7,Bahrain
4,1074,7,5,10.0,5868338.0,1:36.623,201.641,1,race,haas,bahrain,7,Bahrain
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1354,1144,14,16,0.0,0.0,1:25.637,222.002,11,race,haas,yas_marina,3,UAE
1355,1144,12,17,0.0,0.0,1:28.751,214.212,5,race,rb,yas_marina,3,UAE
1356,1144,9,18,0.0,0.0,1:29.482,212.462,130,race,sauber,yas_marina,3,UAE
1357,1144,20,19,0.0,0.0,1:29.411,212.631,5,race,williams,yas_marina,3,UAE


In [ ]:
def convert_time(time_str):
    if pd.isna(time_str) or time_str == '':
        return 0
    try:
        minutes_str, seconds_and_ms_str = time_str.split(':')
        minutes = int(minutes_str)
        minutes_in_ms = minutes * 60 * 1000
        seconds_in_ms = float(seconds_and_ms_str) * 1000
        total_milliseconds = int(minutes_in_ms + seconds_in_ms)
        return total_milliseconds
    except ValueError:
        # Handle cases where split fails or conversion fails (e.g., malformed strings)
        return 0

selected_df['fastestLapTime'] = selected_df['fastestLapTime'].apply(convert_time)



In [92]:
selected_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1359 entries, 0 to 1358
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   raceId           1359 non-null   int64  
 1   start_position   1359 non-null   int64  
 2   finish_position  1359 non-null   int64  
 3   points           1359 non-null   float64
 4   time_in_mil      1359 non-null   float64
 5   fastestLapTime   1359 non-null   int64  
 6   max_speed_mph    1359 non-null   float64
 7   statusId         1359 non-null   int64  
 8   session_type     1359 non-null   object 
 9   constructorRef   1359 non-null   object 
 10  circuitRef       1359 non-null   object 
 11  alt              1359 non-null   int64  
 12  country          1359 non-null   object 
dtypes: float64(3), int64(6), object(4)
memory usage: 148.6+ KB


In [93]:
selected_df.describe().T

,count,mean,std,min,25%,50%,75%,max
raceId,1359.0,1.109534e+03,2.075519e+01,1074.0,1091.5000,1110.000,1.127500e+03,1144.0
start_position,1359.0,9.947020e+00,5.819080e+00,0.0,5.0000,10.000,1.500000e+01,20.0
finish_position,1359.0,1.049301e+01,5.764760e+00,1.0,5.5000,10.000,1.500000e+01,20.0
points,1359.0,5.097130e+00,7.245494e+00,0.0,0.0000,1.000,9.500000e+00,26.0
time_in_mil,1359.0,3.733858e+06,3.032840e+06,0.0,0.0000,5144421.000,5.789092e+06,11012095.0
fastestLapTime,1359.0,8.500609e+04,2.132072e+04,0.0,78925.0000,86876.000,9.659150e+04,125585.0
max_speed_mph,1359.0,2.012024e+02,4.770568e+01,0.0,195.6725,208.912,2.260215e+02,256.1
statusId,1359.0,8.894776e+00,2.337143e+01,1.0,1.0000,1.000,1.100000e+01,141.0
alt,1359.0,2.500294e+02,4.829249e+02,-7.0,7.0000,37.000,2.640000e+02,2227.0
